# Data Cleaning

In [ ]:
import re
import logging


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
RAW_DATA_PATH= "./../data/raw/data.csv"
CLEANED_DATA_PATH="./../data/cleaned/cleaned_data.csv"
QUARANTINE_LOG_PATH="./../data/cleaned/cleaning_quarantined_data.csv"
CLEANING_LOG_REPORT_PATH="./../reports/cleaning_log_report.csv"
CLEANING_LOGGING_PATH="./../reports/cleaning.log"



DISTANCE_COLUMNS = [
    "dist_nearest_school_km", "dist_nearest_hospital_km",
    "dist_nearest_supermarket_km", "dist_nearest_mall_km",
    "dist_nearest_transit_station_km", "dist_nearest_cafe_restaurant_km",
]

COUNT_COLUMNS = [
    "school_count_within_3km", "hospital_count_within_3km",
    "supermarket_count_within_3km", "mall_count_within_3km",
    "transit_station_count_within_3km", "cafe_restaurant_count_within_3km",
]

# Prepare for logging

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    handlers=[
        logging.FileHandler(CLEANING_LOGGING_PATH),   # writes to file
        logging.StreamHandler(),               # prints to console
    ],
)

cleaning_log: list[dict] = []

In [ ]:
def log_cleaning_action(step: str,rule: str,records_affected: int,action: str,rationale: str) -> None:
    """
    Append one cleaning decision to the in-memory cleaning log.

    Args:
        step:The cleaning dimension
        rule:specific rule applied
        records_affected: Number of rows or values changed.
        action: action that was done.
        rationale: Why this action was chosen.
    """
    cleaning_log.append({
        "step": step,
        "rule": rule,
        "records_affected": records_affected,
        "action": action,
        "rationale": rationale,
    })
    logging.info(f"[LOG] {step} | {rule} | {records_affected} records | {action}")

### Load Raw Data

In [ ]:
raw_df = pd.read_csv(RAW_DATA_PATH,low_memory=False)

# Work on a copy, raw_df is never modified
df=raw_df.copy()

print("DATA SHAPE")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
print()

print("COLUMN NAMES")
print(df.columns.tolist())
print()

print("DTTYPES")
print(df.dtypes.value_counts())

# we will remove irrelevant columns to our problem before starting

In [ ]:
irrelevant_columns_to_remove =[
    "listing_id","internal_id","detail_url","title","images_count",	"has_video","video_url","reference","description",
    "agent_id","agent_name","agent_email","agent_is_verified","agent_languages","broker_id","broker_name","broker_email",
    "broker_phone","contact_phone","contact_whatsapp","location_full","contact_email","scraped_at","source","id","url","title.1","scraped_at.1",
    "category", "listing_type","listed_date","subdistrict", "payment_plan","is_direct_from_developer"
]

print("Original number of Columns:" f"{df.shape[1]}")

df_relevance = df.drop(columns=irrelevant_columns_to_remove,errors="ignore")
print("Number of Columns after cleaning:" f"{df_relevance.shape[1]}")

In [ ]:
df=df_relevance.copy()
df.columns

## Step 1: Accuracy

### Rule-based Accuracy Corrections

In [ ]:
df["bedrooms"].value_counts(dropna=False)

In [ ]:
df_cleaned = df.copy()
incorrect_bedrooms_values_count=df[df["bedrooms"]=="studio"].shape[0]
df_cleaned["bedrooms"] = df["bedrooms"].replace("studio", 1).astype(int)

log_cleaning_action(step="Accuracy",rule="Bedrooms: 'studio' replaced with 1",
                    records_affected=incorrect_bedrooms_values_count,action="replaced bedrooms with studio value with 1",
                    rationale="Studio apartments typically have 1 bedroom equivalent")

In [ ]:
df=df_cleaned.copy()

In [ ]:
pd.options.display.max_rows = None

In [ ]:
df[df["price_egp"] > 20_000_000]["district"]

### Quarantine Invalid Records

In [ ]:
# create masks
area_mask = df["area_value"] > 1000
price_mask = df["price_egp"] > 20_000_000
lon_mask = (df["lon"] < 25.0) | (df["lon"] > 35.0)
lat_mask = (df["lat"] < 22.0) | (df["lat"] > 31.0)
very_small_area_mask = df["area_value"] < 40
bedroom_mask = df["bedrooms"] <= 0

rejection_mask = (
    area_mask |     # unrealistic area
    price_mask |    # unrealistic price
    lon_mask |  # invalid longitude
    lat_mask |   # invalid latitude
    very_small_area_mask |  # very small area
    bedroom_mask   # invalid bedrooms
)

df_cleaned =df[~rejection_mask].copy()
df_quarantined =df[rejection_mask].copy()


if area_mask.sum() > 0:
    log_cleaning_action(step="accuracy",rule="area_value > 1000",records_affected=area_mask.sum(),
        action=f"quarantined non realistic {area_mask.sum()} records",
        rationale="non realistic area values likely indicate data entry errors"
    )

if price_mask.sum() > 0:
    log_cleaning_action(step="accuracy",rule="price_egp > 20_000_000",records_affected=price_mask.sum(),
        action=f"quarantined non realistic {price_mask.sum()} records",
        rationale="non realistic price values likely indicate data entry errors"
    )

if lon_mask.sum() > 0:
    log_cleaning_action(step="accuracy",rule="lon out of range",records_affected=lon_mask.sum(),
        action=f"quarantined invalid {lon_mask.sum()} records",
        rationale="longitude values outside of Egypt's geographic range"
    )

if lat_mask.sum() > 0:
    log_cleaning_action(step="accuracy",rule="lat out of range",records_affected=lat_mask.sum(),
        action=f"quarantined invalid {lat_mask.sum()} records",
        rationale="latitude values outside of Egypt's geographic range"
    )    

if very_small_area_mask.sum() > 0:
    log_cleaning_action(step="accuracy",rule="area_value < 40",records_affected=very_small_area_mask.sum(),
        action=f"quarantined very small area {very_small_area_mask.sum()} records",
        rationale="very small area values likely indicate data entry errors or non-residential properties"
    )

if bedroom_mask.sum() > 0:
    log_cleaning_action(step="accuracy",rule="bedrooms <= 0",records_affected=bedroom_mask.sum(),
        action=f"quarantined invalid bedrooms {bedroom_mask.sum()} records",
        rationale="zero or negative bedroom counts are not realistic for residential properties"
    )


df_quarantined["rejection_reason"] = (
    df[rejection_mask].apply(
        lambda row: '; '.join([
            'area_value > 1000' if row['area_value'] > 1000 else '',
            'price_egp > 20M' if row['price_egp'] > 20_000_000 else '',
            'invalid_lon' if (row['lon'] < 25.0 or row['lon'] > 35.0) else '',
            'invalid_lat' if (row['lat'] < 22.0 or row['lat'] > 31.0) else '',
            'invalid_bedrooms' if row['bedrooms'] <= 0 else '',
        ]).strip('; '),
        axis=1
    )
)

log_cleaning_action(step="accuracy",rule="total_quarantined",records_affected=rejection_mask.sum(),
    action=f"quarantined {rejection_mask.sum()} records with reasons logged",
    rationale="Allows for later review and potential recovery of records if needed"
)


In [ ]:
df = df_cleaned.copy()

# consistency step

In [ ]:
def show_value_counts(df,columns=None):
    for col in columns:
        print(f"\nColumn: {col}")
        print(df[col].value_counts(dropna=False))

### property_type

In [ ]:
df["property_type"].value_counts()

In [ ]:
mapping = {
    "apartments": "apartment",
}   
df["property_type"]=df["property_type"].str.lower().replace(mapping)

log_cleaning_action(step="consistency",rule="standardize_property_types",records_affected=mapping["apartments"],
    action=f"standardized {mapping['apartments']} records from 'apartments' to 'apartment'",
    rationale="for consistency, 'apartment' is the more common term and helps unify the category"
)


**if is just one value drop the column**

In [ ]:
if df["property_type"].nunique() ==1:
    df.drop(columns=["property_type"],inplace=True)

    log_cleaning_action(step="consistency",rule="drop_property_type_if_single_value",records_affected=0,
        action=f"dropped 'property_type' column as it has only one unique value",
        rationale="for consistency, the column is redundant when it contains only a single value"
    )


### offering_type
-   for-sale                
-   Residential for Sale

convert to :<br>
-   for-sale

In [ ]:
df["offering_type"].value_counts()

In [ ]:
mapping = {
    'Residential for Sale': 'for-sale',
    'for-sale': 'for-sale'
}

df["offering_type"]=df["offering_type"].replace(mapping)

log_cleaning_action(step="consistency",rule="standardize_offering_types",records_affected=mapping['Residential for Sale'],
    action=f"standardized {mapping['Residential for Sale']} records from 'Residential for Sale' to 'for-sale'",
    rationale="for consistency, 'for-sale' is more concise and unifies the category"
)

In [ ]:
df["offering_type"].value_counts()

In [ ]:
if df["offering_type"].nunique() ==1:
    df.drop(columns=["offering_type"],inplace=True)

    log_cleaning_action(step="consistency",rule="drop_offering_type_if_single_value",records_affected=0,
        action=f"dropped 'offering_type' column as it has only one unique value",
        rationale="for consistency, the column is redundant when it contains only a single value"
    )


### completion_status
- completed             
- under-construction     
- off_plan_primary       
- completed_primary      
- off_plan                

convert to :<br>
- completed
- under-construction
- off_plan


In [ ]:

mapping = {
    'completed_primary': 'completed',
    'off_plan_primary': 'off_plan'
}

df["completion_status"]=df["completion_status"].copy().replace(mapping)

log_cleaning_action(step="consistency",rule="standardize_completion_status",records_affected=mapping['off_plan_primary'],
    action=f"standardized {mapping['off_plan_primary']} records from 'off_plan_primary' to 'off_plan'",
    rationale="for consistency, 'off_plan' is more concise and unifies the category"    
)


In [ ]:
df["completion_status"].value_counts()

### price_period
**one unique value so no need to keep it**

In [ ]:
df["price_period"].value_counts()

if df["price_period"].nunique() ==1:
    df.drop(columns=["price_period"],inplace=True)

    log_cleaning_action(step="consistency",rule="drop_price_period_if_single_value",records_affected=0,
        action=f"dropped 'price_period' column as it has only one unique value",    
        rationale="for consistency, the column is redundant when it contains only a single value"
    )

### price_currency
**one unique value so no need to keep it**

In [ ]:
df["price_currency"].value_counts()
if df["price_currency"].nunique() ==1:
    df.drop(columns=["price_currency"],inplace=True)
    log_cleaning_action(step="consistency",rule="drop_price_currency_if_single_value",records_affected=0,
        action=f"dropped 'price_currency' column as it has only one unique value",  
        rationale="for consistency, the column is redundant when it contains only a single value"
    )
    

### town

In [ ]:
df["town"].nunique()

In [ ]:
df["town"].value_counts()

In [ ]:
# remove the word city in town names
df["town"] = df["town"].str.replace(r'\bCity\b', '', regex=True).str.strip().str.title()

log_cleaning_action(step="consistency",rule="standardize_town_names",records_affected=df["town"].str.contains(r'\bCity\b', regex=True).sum(),
    action=f"standardized town names by removing 'City' suffix and applying title case",
    rationale="for consistency, removing 'City' suffix unifies town names and title case improves readability"
)


### district

In [ ]:


# lowercase + remove extra spaces
df["district"] = (
    df["district"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

log_cleaning_action(step="consistency",rule="standardize_district_names",records_affected=df["district"].notna().sum(),
    action=f"standardized district names by lowercasing and removing extra spaces",
    rationale="for consistency, lowercasing and removing extra spaces unifies district names"
)

df["district"] = (
    df["district"]
    .str.replace(r"[^\w\s]", "", regex=True)   # remove punctuation
)

log_cleaning_action(step="consistency",rule="remove_punctuation_from_districts",records_affected=df["district"].notna().sum(),
    action=f"removed punctuation from district names",  
    rationale="for consistency, removing punctuation unifies district names"
)

In [ ]:
df['district'].nunique()

### area_unit

In [ ]:
df['area_unit'].value_counts()

In [ ]:
df['area_unit'] = df['area_unit'].str.lower().str.strip()

log_cleaning_action(step="consistency",rule="standardize_area_unit",records_affected=df['area_unit'].notna().sum(),
    action=f"standardized 'area_unit' by lowercasing and stripping spaces",
    rationale="for consistency, lowercasing and stripping spaces unifies area unit values"
)


In [ ]:
if df['area_unit'].nunique() ==1:
    df.drop(columns=['area_unit'],inplace=True)
    log_cleaning_action(step="consistency",rule="drop_area_unit_if_single_value",records_affected=0,
        action=f"dropped 'area_unit' column as it has only one unique value",
        rationale="for consistency, the column is redundant when it contains only a single value"
    )

### furnished

In [ ]:
df['furnished'].value_counts()

In [ ]:
mapping = {'NO': 'unfurnished', 'YES': 'furnished','PARTLY':'partly'}
df['furnished'] = df['furnished'].replace(mapping)
log_cleaning_action(step="consistency",rule="standardize_furnished",records_affected=df['furnished'].isin(mapping.keys()).sum(),
    action=f"standardized 'furnished' values by mapping {mapping}",
    rationale="for consistency, unifying furnished status values improves clarity and analysis"
)

In [ ]:
df['furnished'].value_counts()

### col with one unique value
- is_verified 
-  is_new_construction 
-  rera

**are only one unique value so we can drop them**

In [ ]:
print(df["is_new_construction"].value_counts())
print(df["is_verified"].value_counts())
print(df["rera"].value_counts())

In [ ]:
if df["is_verified"].nunique() ==1:
    df.drop(columns=['is_verified'], inplace=True)  

    log_cleaning_action(step="consistency",rule="drop_is_verified_if_single_value",records_affected=0,
        action=f"dropped 'is_verified' column as it has only one unique value",
        rationale="for consistency, the column is redundant when it contains only a single value"
    )
    
if df["is_new_construction"].nunique() ==1:
    df.drop(columns=['is_new_construction'], inplace=True) 

    log_cleaning_action(step="consistency",rule="drop_is_new_construction_if_single_value",records_affected=0,
        action=f"dropped 'is_new_construction' column as it has only one unique value",
        rationale="for consistency, the column is redundant when it contains only a single value"
    )

    
if df["rera"].nunique() <=1:
    df.drop(columns=['rera'], inplace=True) 
    log_cleaning_action(step="consistency",rule="drop_rera_if_single_or_no_value",records_affected=0,
        action=f"dropped 'rera' column as it has only one or no unique value",
        rationale="for consistency, the column is redundant when it contains only a single or no value"
    ) 

### amenities

In [ ]:
for unique_value in df['amenities'].unique():
    print({unique_value})

In [ ]:
import pandas as pd
import numpy as np

def standardize_amenities(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

   
    x = x.strip("{}")
    x = x.strip('"')
    x = x.strip("'")

   
    amenities = [item.strip().lower() for item in x.split("|")]

  
    amenities = [a for a in amenities if a]

    
    amenities = sorted(set(amenities))

    
    return " | ".join(amenities)

df["amenities"] = df["amenities"].apply(standardize_amenities)

## Step 2: Consistency

### Coerce Types for Numerical , Bool and Categorical Columns

In [ ]:
df = df.replace(
    ["nan", "none", "None", "", "null", "NULL"],
    np.nan
)
numeric_cols = [
    "price_egp",
    "lat",
    "lon",
    "area_value",
    "dist_nearest_school_km",
    "school_count_within_3km",
    "dist_nearest_hospital_km",
    "hospital_count_within_3km",
    "dist_nearest_supermarket_km",
    "supermarket_count_within_3km",
    "dist_nearest_mall_km",
    "mall_count_within_3km",
    "dist_nearest_transit_station_km",
    "transit_station_count_within_3km",
    "dist_nearest_cafe_restaurant_km",
    "cafe_restaurant_count_within_3km"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")


df["bedrooms"] = pd.to_numeric(df["bedrooms"], errors="coerce").astype("Int64")
df["bathroom"] = (
    df["bathroom"]
    .replace({
        "none": np.nan,
        "7+": 7
    })
)

df["bathroom"] = pd.to_numeric(
    df["bathroom"],
    errors="coerce"
).astype("Int64")

bool_cols = [
    "is_premium",
    "is_featured",
    "is_exclusive"
]

for col in bool_cols:
    df[col] = df[col].astype("boolean")

In [ ]:
df["lat"] = df["lat"].round(6)
df["lon"] = df["lon"].round(6)

In [ ]:
cat_cols = [
    "completion_status",
    "city",
    "town",
    "district",
    "furnished",
    "listing_level",
    "amenities",
]

for col in cat_cols:
    df[col] = df[col].astype("category")

In [ ]:
df_consistency = df.copy()

In [ ]:
df=df_consistency.copy()

## Step 3: Completeness

### Reporting Missingness

In [ ]:
def replace_placeholder_strings(dataframe: pd.DataFrame, columns: list) -> int:
    """
    Replace placeholder strings in specified columns with NaN.
    
    Args:
        dataframe: DataFrame to clean
        columns: List of column names to process
        
    Returns:
        Total count of replacements made
    """

    PLACEHOLDER_STRINGS = [
        "", " ", "?", "N/A", "n/a", "Unknown", "unknown",
        "none", "None", "null", "Null", "missing", "Missing", "-",
    ]


    total_replacements = 0
    for col in columns:
        mask = dataframe[col].astype(str).str.strip().isin(PLACEHOLDER_STRINGS)
        replacements = mask.sum()
        dataframe.loc[mask, col] = np.nan
        total_replacements += replacements
    return total_replacements

# Apply to all string columns
string_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
replaced_count = replace_placeholder_strings(df, string_cols)
replaced_count

In [ ]:
def audit_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Produce a full missing value report sorted by missingness descending.

    Args:
        df: Input DataFrame.

    Returns:
        DataFrame with columns: column, missing_count, missing_pct, dtype.
    """
    missing_counts = df.isna().sum()
    missing_pcts   = (df.isna().mean() * 100).round(2)

    report = pd.DataFrame({
        "column":        missing_counts.index,
        "missing_count": missing_counts.values,
        "missing_pct":   missing_pcts.values,
        "dtype":         df.dtypes.values,
    })

    return report[report["missing_count"] > 0].sort_values("missing_pct", ascending=False)


missing_report = audit_missing_values(df)
print(f"Columns with missing values: {len(missing_report)}")
missing_report

In [ ]:
df["furnished"].value_counts(dropna=False)

what should we do in amentities

In [ ]:
cols_to_drop = ["is_exclusive",  # about 80% missing and value exist in listing_level column which is more complete
                "amenities"  # about 75% missing and any imputation would introduce false information or noise
                ]
df = df.drop(columns=cols_to_drop)
log_cleaning_action(
    step="Completeness",
    rule=f"Drop {', '.join(cols_to_drop)} cols high-missingness",
    records_affected=len(df),
    action=f"Dropped columns: {cols_to_drop}",
    rationale="Columns with >50% missing AND not required as model features",
)
print(f"Shape after column drops: {df.shape}")
print(f"Dropped {len(cols_to_drop)} columns")

In [ ]:
def fill_district_with_mode(df):
    number_of_missing_districts = int(df["district"].isna().sum())

    if number_of_missing_districts == 0:
        return df,0
        
    df["district"] = df["district"].cat.add_categories(["Unknown"])
    group_mode = df.groupby(['city', 'town'])['district'].transform(
        lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
    )
    df['district'] = df['district'].fillna(group_mode)

    city_mode = df.groupby('city')['district'].transform(
        lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
    )
    df['district'] = df['district'].fillna(city_mode)

    df['district'] = df['district'].fillna('Unknown')

    return df,number_of_missing_districts

In [ ]:
df,number_of_missing_districts=fill_district_with_mode(df)
if number_of_missing_districts > 0:
    log_cleaning_action(
        step="completeness",
        rule="fill_district_with_mode",
        action="fill_district_with_mode",
        records_affected=number_of_missing_districts,
        rationale="Filled missing 'district' values using mode of 'city' and 'town' groups if not empty, " \
                    "then by 'city' groups, then set remaining to 'Missing' if no mode found",
    )

In [ ]:
n_furnished_missing = int(df["furnished"].isna().sum())

if n_furnished_missing > 0:
    df["furnished"] = df["furnished"].cat.add_categories(["Missing"])
    df["furnished"] = df["furnished"].fillna("Missing")

    log_cleaning_action(
        step="Completeness",
        rule="furnished: fill NaN with 'Missing' category",
        records_affected=n_furnished_missing,
        action="fillna('Missing')",
        rationale=(
            "Missingness is informative — agents who don't disclose furnished status "
            "may represent a distinct listing pattern. 'Missing' lets the model learn from it."
        ),
    )

In [ ]:
number_missing_completion = int(df["completion_status"].isna().sum())
df = df.dropna(subset=['completion_status'])
log_cleaning_action(
    step="Completeness",
    rule="Drop rows with missing completion status",
    records_affected=number_missing_completion,
    action="dropna(subset=['completion'])",
    rationale=(
        "Completion status is a critical feature for modeling. "
        "Missingness is relatively low, so dropping is preferable to imputation since very few rows are affected."
    ),
)

In [ ]:
number_missing_bathrooms = int(df["bathroom"].isna().sum())
df = df.dropna(subset=['bathroom'])
log_cleaning_action(
    step="Completeness",
    rule="Drop rows with missing bathroom count",
    records_affected=number_missing_bathrooms,
    action="dropna(subset=['bathroom'])",
    rationale=(
        "Bathroom count is a critical feature for modeling. "
        "Missingness is relatively low, so dropping is preferable to imputation since very few rows are affected."
    ),
)

In [ ]:
missing_report = audit_missing_values(df)
print(f"Columns with missing values: {len(missing_report)}")
missing_report

In [ ]:
completeness_df=df.copy()

In [ ]:
df=completeness_df.copy()

## Step 4: Uniqueness

### Removing Duplicates

In [ ]:
print(df.shape)

In [ ]:
count_of_duplicates=df.duplicated().sum()
df.drop_duplicates(inplace=True)
n_after = df.shape[0]
log_cleaning_action(
    step="Uniqueness",
    rule="drop_exact_duplicates",
    records_affected=count_of_duplicates,
    action="drop_duplicates()",
    rationale=f"Removed {count_of_duplicates} exact duplicate rows to ensure uniqueness of listings."
)
logging.info(f"Shape after dropping duplicates: {df.shape} (dropped {count_of_duplicates} rows)")

In [ ]:
uniqueness_df=df.copy()

In [ ]:
df=uniqueness_df.copy()

## Step 5: Outliers

### Detect Outliers Using IQR

In [ ]:
def check_number_IQR_outliers(df,column):
    q1=df[column].quantile(0.25)
    q3= df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return sum((df[column] < lower_bound) | (df[column] > upper_bound))

In [ ]:
def apply_clipping(df,column,lower_perc,upper_perc):
    df_clipped=df.copy()

    lower_cap = df_clipped[column].quantile(lower_perc)
    upper_cap = df_clipped[column].quantile(upper_perc)
    number_of_clipped_values = sum((df_clipped[column] < lower_cap) | (df_clipped[column] > upper_cap))
    df_clipped[column] = df_clipped[column].clip(lower=lower_cap, upper=upper_cap)

    return df_clipped, number_of_clipped_values

### Cap columns at percentile thresholds

In [ ]:
print(df["area_value"].mean(), df["area_value"].median())
print(f"difference: {df['area_value'].mean() - df['area_value'].median()}")
print("difference between mean and median indicates they are almost symmetrically distributed with a slight right skew")

In [ ]:
df["area_value"].hist()

In [ ]:
df,number_of_clipped_values=apply_clipping(df,"area_value",0.01,0.95)
log_cleaning_action(
    step="outlier",
    rule="clip_area_value_outliers",
    records_affected=number_of_clipped_values,
    action="Clipped area_value to 1st and 95th percentiles",
    rationale=(
        "Clipping extreme outliers in 'area_value' reduces their disproportionate influence on the model while preserving the overall distribution shape. "
        "The mean and median are close, indicating a relatively symmetric distribution with some right skew, so clipping helps mitigate the impact of extreme high values."
    )
)

In [ ]:
for column in DISTANCE_COLUMNS:
    num_outliers=check_number_IQR_outliers(df,column)
    print(f"Number of IQR outliers in '{column}': {num_outliers}")

In [ ]:
for column in COUNT_COLUMNS:
    num_outliers=check_number_IQR_outliers(df,column)
    print(f"Number of IQR outliers in '{column}': {num_outliers}")

In [ ]:
for column in DISTANCE_COLUMNS:
    print(f"\nBefore clipping {column}:")
    print(df[column].mean(), df[column].median())
    print(f"difference: {df[column].mean() - df[column].median()}")

In [ ]:
for column in DISTANCE_COLUMNS:
    data = df[column].dropna()

    mean_val = data.mean()
    median_val = data.median()

    plt.figure(figsize=(8, 4))
    plt.hist(data, bins=30)

    plt.axvline(mean_val, linestyle='--', label='mean')
    plt.axvline(median_val, linestyle='-', label='median')

    plt.title(f"Distribution of {column}")
    plt.legend()
    plt.show()

In [ ]:
for column in COUNT_COLUMNS:
    print(f"\nBefore clipping {column}:")
    print(df[column].mean(), df[column].median())
    print(f"difference: {df[column].mean() - df[column].median()}")

In [ ]:
for column in COUNT_COLUMNS:
    data = df[column].dropna()

    mean_val = data.mean()
    median_val = data.median()

    plt.figure(figsize=(8, 4))
    plt.hist(data, bins=30)

    plt.axvline(mean_val, linestyle='--', label='mean')
    plt.axvline(median_val, linestyle='-', label='median')

    plt.title(f"Distribution of {column}")
    plt.legend()
    plt.show()

In [ ]:
CLIPPING_COLUMNS_DISTANCE=["dist_nearest_transit_station_km"]
LOG_COLUMNS_DISTANCE=[
    "dist_nearest_school_km", "dist_nearest_hospital_km",
    "dist_nearest_supermarket_km", "dist_nearest_mall_km",
    "dist_nearest_cafe_restaurant_km",
]


CLIPPING_COLUMNS_COUNT=["mall_count_within_3km","transit_station_count_within_3km"]
LOG_COLUMNS_COUNT=["school_count_within_3km", "hospital_count_within_3km",
    "supermarket_count_within_3km", "cafe_restaurant_count_within_3km",
]


In [ ]:
CLIPPING_COLUMNS=CLIPPING_COLUMNS_DISTANCE + CLIPPING_COLUMNS_COUNT
LOG_COLUMNS=LOG_COLUMNS_DISTANCE + LOG_COLUMNS_COUNT

In [ ]:
assert len(CLIPPING_COLUMNS) + len(LOG_COLUMNS) == len(DISTANCE_COLUMNS) + len(COUNT_COLUMNS)

In [ ]:
for column in CLIPPING_COLUMNS:
    df,clipped_numbers=apply_clipping(df,column,0.0,0.95)
    log_cleaning_action(
        step="outlier",
        rule=f"clip_{column}_outliers",
        records_affected=clipped_numbers,
        action=f"Clipped '{column}' to 95th percentile",
        rationale=(
            f"Clipping extreme outliers in '{column}' reduces their disproportionate"
            "influence on the model while preserving the overall distribution shape. "
        )
    )

In [ ]:
for column in LOG_COLUMNS:
    # apply log transformation
    df[column] = np.log1p(df[column])

    log_cleaning_action(
        step="outlier",
        rule=f"log_transform_{column}",
        records_affected=df[column].notna().sum(),
        action=f"Applied log1p transformation on '{column}'",
        rationale=(
            f"Log transformation reduces right skewness in '{column}' and compresses extreme values, "
            "making the distribution more stable and less sensitive to outliers while preserving ordering."
        )
    )

In [ ]:
cleaned_df=df.copy()

## Save Outputs

In [ ]:
df_quarantined.to_csv(QUARANTINE_LOG_PATH, index=False)
cleaned_df.to_csv(CLEANED_DATA_PATH, index=False)
cleaning_report_df = pd.DataFrame(cleaning_log)
cleaning_report_df.to_csv(CLEANING_LOG_REPORT_PATH, index=False)